# 04-04 트랜스포머 블록 만들기

런타임 유형을 GPU로 바꾼 뒤 첫 셀부터 차례로 실행하세요.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
print("device:", device)

## 1. 04-02의 멀티헤드 어텐션

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def split(self, t):
        B, T, _ = t.shape
        return t.view(B, T, self.n_heads, self.d_head).transpose(1, 2)  # (B, H, T, d_head)

    def forward(self, x, mask=None):
        B, T, d_model = x.shape
        q, k, v = self.split(self.q_proj(x)), self.split(self.k_proj(x)), self.split(self.v_proj(x))
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.d_head)  # (B, H, T, T)
        if mask is not None:
            scores = scores.masked_fill(~mask, float("-inf"))
        self.weights = F.softmax(scores, dim=-1)  # (B, H, T, T)
        out = (self.weights @ v).transpose(1, 2).reshape(B, T, d_model)  # (B, T, d_model)
        return self.out_proj(out)

## 2. 피드포워드 레이어

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),  # (B, T, d_model) -> (B, T, d_ff)
            nn.GELU(),
            nn.Linear(d_ff, d_model),  # (B, T, d_ff) -> (B, T, d_model)
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

## 3. 트랜스포머 블록

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # x: (B, T, d_model)
        x = x + self.dropout(self.attn(self.norm1(x), mask))  # 토큰끼리 정보 섞기
        x = x + self.ff(self.norm2(x))  # 토큰마다 따로 계산하기
        return x  # (B, T, d_model)

block = TransformerBlock(d_model=32, n_heads=4, d_ff=128)
x = torch.randn(2, 10, 32)  # (B=2, T=10, d_model=32)
print(block(x).shape)

for name, m in block.named_children():
    print(f"{name:8s} 파라미터 {sum(p.numel() for p in m.parameters()):5d}개")

## 4. nn.TransformerEncoderLayer와 비교하기

In [ ]:
torch_block = nn.TransformerEncoderLayer(
    d_model=32, nhead=4, dim_feedforward=128, activation="gelu", norm_first=True, batch_first=True
)
print("직접 만든 블록:", sum(p.numel() for p in block.parameters()))
print("PyTorch 블록:  ", sum(p.numel() for p in torch_block.parameters()))

## 5. 인코더: 임베딩 + 위치 + 블록 N개

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_heads=4, n_layers=2, d_ff=256, max_len=128, dropout=0.1):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.blocks = nn.ModuleList([TransformerBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, ids, mask=None):
        # ids: (B, T)
        T = ids.shape[1]
        x = self.tok_emb(ids) + self.pos_emb(torch.arange(T, device=ids.device))  # (B, T, d_model)
        for block in self.blocks:
            x = block(x, mask)  # (B, T, d_model)
        return self.norm(x)  # (B, T, d_model)

enc = Encoder(vocab_size=10)
ids = torch.randint(0, 10, (2, 7))  # (2, 7)
print(ids.shape, "->", enc(ids).shape)

## 6. 작은 문제로 확인하기: 문장에서 가장 큰 숫자 찾기

In [ ]:
class MaxTagger(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder(vocab_size=10, d_model=64, n_heads=4, n_layers=2, d_ff=128, max_len=16, dropout=0.0)
        self.head = nn.Linear(64, 10)

    def forward(self, ids):
        return self.head(self.encoder(ids))  # (B, T, 10)

def make_data(batch, T=8):
    ids = torch.randint(0, 10, (batch, T))  # (B, T)
    target = ids.max(dim=1, keepdim=True).values.expand(-1, T)  # (B, T), 모든 위치의 정답이 최댓값
    return ids.to(device), target.to(device)

torch.manual_seed(0)
model = MaxTagger().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
losses = []
for step in range(1500):
    ids, target = make_data(128)
    logits = model(ids)  # (B, T, 10)
    loss = F.cross_entropy(logits.reshape(-1, 10), target.reshape(-1))  # (B*T, 10), (B*T,)
    opt.zero_grad()
    loss.backward()
    opt.step()
    losses.append(loss.item())
    if step % 300 == 0:
        print(f"step {step:4d}: loss={loss.item():.4f}")

model.eval()
with torch.no_grad():
    ids, target = make_data(1000)
    acc = (model(ids).argmax(-1) == target).float().mean().item()
print(f"정확도: {acc:.3f}")

In [ ]:
with torch.no_grad():
    sample = torch.tensor([[3, 1, 7, 2, 0, 5, 1, 2]], device=device)  # (1, 8)
    pred = model(sample).argmax(-1)
    w = model.encoder.blocks[-1].attn.weights[0].mean(0).cpu()  # (8, 8), 헤드 평균
print("입력:", sample.tolist()[0])
print("예측:", pred.tolist()[0])

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].plot(losses, color="tab:red")
axes[0].set_title("loss")
axes[0].set_xlabel("step")
axes[1].imshow(w, cmap="Oranges")
axes[1].set_xticks(range(8), sample.tolist()[0])
axes[1].set_yticks(range(8), sample.tolist()[0])
axes[1].set_xlabel("key token")
axes[1].set_ylabel("query token")
axes[1].set_title("last block attention (mean of heads)")
plt.show()